# Prompt Engineering
## Part 1

<div class="slide-title">

# Prompt Engineering
    
</div>    

## LLMs

- Large Language Models are trained in two stages:
    - **Pretraining** (self-supervised): predict the next word over huge amounts of text — no human labels needed
    - **Alignment** (instruction-tuning / RLHF): human feedback teaches the model to follow instructions and be helpful
- The result behaves like a model that's very good at "guessing the next word"

<img src="../images/prompt_engineering/llm_example.png" width=500 position="center">

Notes: "Supervised" was the old shorthand, but it's a bit misleading — pretraining needs no human-labeled examples at all, it just predicts the next token on raw text at massive scale. The "supervision" signal is the text itself. The instruction-following/chat behavior people actually interact with comes from a second stage (instruction-tuning and RLHF) layered on top. Worth clarifying if a curious student pushes back on "supervised."

### Generating text

In order to generate new text, the model needs a starting point

- Provide some initial text for the LLM to complete
- We call this a **prompt**

## How prompts work

### First, a little game 😄

https://gandalf.lakera.ai/baseline

Try to trick the AI into revealing its secret password.

<img src="../images/prompt_engineering/gandalf_game_qr.png" width=150 position="center">

### What Just Happened?

<div class="alert alert-block alert-warning">
You just performed a <b>prompt injection</b> — crafting input that overrides or bypasses a model's original instructions.
</div>

- Gandalf's hidden system prompt says "never reveal the password"
- Your job was to write a user message that gets around it anyway
- Real applications face this too: chatbots, coding assistants, and agents can all be manipulated this way

Notes: Great segue into why prompt structure matters (next slide). Ask: "What tricks did you use? Roleplay? Asking indirectly? Claiming authority?" Mention that keeping instructions in a separate system message (rather than trusting user input) is a first line of defense — but not a complete one. Prompt injection is still an open problem, and it matters even more once agents can take real actions (see the Agents session).

### Prompt structure

A typical LLM prompt structure consists of the following messages:

- **System message:**
    - Unique, applies to all instructions on the model
    - Is defined only once based on our requirements

- **User message:**
    - Has the question that needs to be answered
    - Expecting a new message each time we run inference

- **(Optional) Assistant message:**
    - Includes the expected response from the LLM
    - Can include prior model responses in multi-turn conversations or examples; also used for training/fine-tuning



**In practice**, this looks like a list of role-tagged messages sent to the model:
```
system: You are a helpful assistant that answers in one sentence.
user: What causes seasons on Earth?
assistant: Seasons are caused by the tilt of Earth's axis relative to its orbit.
user: And why are they reversed in the southern hemisphere?
```

Similar role-based message structures are common across major providers, but role names and system-message handling vary by provider.

### Why Being Specific Matters

<div class="group">
  <div class="text_70">

❌ **Vague:**
```
Write something about dogs.
```

✅ **Specific:**
```
Write a 3-sentence, upbeat product
description for a dog training app,
aimed at first-time puppy owners.
```
  </div>
</div>

The more clearly you specify instruction, context, input, and format — the more reliable the output.

## Best Practices for Effective Prompts

### Tips That Actually Work

- **Be specific and clear** — vague prompts get vague answers
- **Use delimiters** — separate instructions from content with quotes, triple backticks, or tags
- **Give the model a role** — "You are an experienced technical writer..."
- **Give it room to reason** — ask it to think step by step before answering (see Chain-of-Thought, next)
- **Specify the output format** — "Answer in a table", "Respond only with JSON"
- **Iterate** — treat your first prompt as a draft, not a final answer

Notes: Live-demo one of these with the group — take a vague prompt they suggest and improve it together using 2-3 of these tips. This is more memorable than just reading the list.

## Prompt Templates

### Zero-shot Prompting


**Prompt:**
```
Classify the text into neutral, negative or positive. 
Text: I think the vacation is okay.
Sentiment:
```
**Output:**
```
Neutral
```

### Few-shot Prompting

**Prompt:**
```
Classify the text into neutral, negative or positive. 
Follow these examples:

Example 1: 
Text: I think the movie is very bad
Sentiment: Negative

Example 2:
Text: Today was a good day!
Sentiment: Positive

Example 3:
Text: Today we learnt about Machine Learning
Sentiment: Neutral

Now classify the following text:

Text: I think the vacation is okay.
Sentiment:
```
**Output:**
```
Neutral
```

⚠️ More examples means more tokens (cost/latency), and biased examples can bias the output — a few well-chosen examples usually beat many mediocre ones.

### Chain-of-Thought (CoT) Prompting

<img src="../images/prompt_engineering/cot_example.png" width=800 position="center">

### Zero-shot CoT: An Even Simpler Trick

You don't always need examples — just ask the model to reason first.

**Prompt:**
```
Q: A juggler can juggle 16 balls. Half of the balls
are golf balls, and half of the golf balls are blue.
How many blue golf balls are there?

Let's think step by step.
```
**Output:**
```
There are 16 balls in total. Half are golf balls,
so there are 8 golf balls. Half of the golf balls
are blue, so there are 4 blue golf balls.
```

Adding just **"Let's think step by step"** measurably improves accuracy on reasoning tasks — no examples required.

Notes: This is the "zero-shot CoT" trick from Kojima et al. (2022) — cited in References. Emphasize that it adds no few-shot examples, though generating the extra reasoning can increase output tokens, latency, and cost; validate whether it helps on the target task.

### Structured Output

Sometimes you don't want prose back — you want data your code can use.

**Prompt:**
```
Extract the name, age, and city from the text below.
Return only valid JSON with keys: name, age, city.

Text: "Maria is 29 and lives in Berlin."
```
**Output:**
```json
{
  "name": "Maria",
  "age": 29,
  "city": "Berlin"
}
```

<div class="alert alert-block alert-warning">
Most LLM providers now also offer dedicated <b>structured/JSON output modes</b> that constrain the model to a schema — more reliable than prompting alone.
</div>

This is exactly what powers **tool use** and **agents** (next session): the model's output needs to be parseable, not just readable.

## Wrapping Up

### Prompting: Pros and Cons

#### PROS:
- Fast to iterate — no training required
- Works across a huge range of tasks
- Cheap to experiment with

#### CONS:
- Brittle — small wording changes can change the output
- Can't teach the model new or private knowledge → **next up: RAG**
- Can't take actions in the world on its own → **next up: AI Agents**

## References

* [Prompt Engineering Guide](https://www.promptingguide.ai/)
* [Anthropic: Prompt Engineering Overview](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview)
* [OpenAI: Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
* [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)
* [Large Language Models are Zero-Shot Reasoners](https://arxiv.org/abs/2205.11916)